# Day 1 · 02 · Gold Semantic Layer for Power BI Developers

![Star schema vs flat table](../../assets/images/star_schema_vs_flat_table.png)

This notebook is an instructor-led demo for experienced Power BI developers.
It focuses on the Databricks side of the semantic model: how to shape Gold
tables, views and aggregates so Power BI stays thin, fast and governable.

KPI definitions are still included, but they are not the main topic. They are
one part of a larger Gold contract that covers grain, relationships, ownership,
quality checks, refresh expectations and performance-oriented aggregates.

Hands-on work is separated into `notebooks/workshops/`: one workshop for Day 1
and one workshop for Day 2.

## Demo Scenario

The Power BI team already knows how to build reports. The problem is that every
report has started to carry its own data-engineering logic:

- duplicated DAX measures,
- Power Query transformations that should be shared,
- flat tables that are convenient for one report but expensive for the next,
- DirectQuery visuals that fan out into many warehouse queries,
- unclear ownership of KPI definitions,
- no lineage from dashboard number back to approved Gold logic.

The demo answer is to build a Databricks Gold semantic layer that Power BI can
consume predictably.

## Learning Objectives

By the end of this demo you can:

- translate Power BI reporting needs into a Databricks Gold contract,
- decide what belongs in Gold vs Power BI measures / Power Query,
- choose between fact/detail table, dimension table, aggregate table and view,
- confirm fact grain before relationships, slicers and measures are designed,
- build Import-friendly and DirectQuery-friendly Gold objects,
- define KPI with business and SQL definitions when the metric must be shared,
- catch bad joins, metric drift and hidden DAX duplication,
- decide whether quality issues belong in Silver, Gold or the report layer,
- document model decisions in a way another BI developer can reuse.

## Setup

Initialize the environment, resolve the participant catalog and expose the shared variables used by this notebook.

In [ ]:
%run ../../setup/00_setup

### Configuration

Display the active RetailHub catalog context and validate prerequisite objects before the demo starts.

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

Fail fast if an upstream setup or demo notebook has not created the required tables.

In [ ]:
required_tables = [
    f"{GOLD}.fact_sales_dashboard",
    f"{GOLD}.dim_date",
    f"{GOLD}.dim_product",
    f"{GOLD}.dim_customer",
]

missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    print("[BLOCKED] Missing required tables:")
    for t in missing:
        print("  -", t)
    print()
    print("Run this notebook first: notebooks/demo/day1_01_lakehouse_sql_gold.ipynb")
    raise Exception("Pre-check failed: missing prerequisite tables. Run \"notebooks/demo/day1_01_lakehouse_sql_gold.ipynb\" first.")

print("[OK] Pre-check passed, all required tables exist:")
for t in required_tables:
    print("  -", t)

## 1. Power BI developer pain points

Experienced Power BI developers usually do not need another basic KPI lesson.
They need a better upstream contract.

| Pain point in Power BI | Gold-layer response in Databricks |
|---|---|
| same DAX copied into many reports | approved KPI table or metric contract |
| slow refresh on wide detail table | aggregate table for summary pages |
| DirectQuery sends too many expensive SQL queries | narrow Gold object, low-cardinality slicers, predictable filters |
| ambiguous relationships | explicit fact/dimension grain and keys |
| Power Query cleans business data per report | Silver/Gold transformation owned upstream |
| business asks "which revenue is true?" | reconciliation and owner-approved definition |
| no lineage for report data | Unity Catalog comments, lineage and table ownership |

The objective is not to replace Power BI modelling skills. The objective is to
remove reusable data logic from individual PBIX files and place it in governed
Gold objects.

## 2. Gold semantic contract

![Gold business value](../../assets/images/gold_business_value.png)

A Gold object used by Power BI should have an explicit contract:

| Contract area | Question to answer before Power BI connects |
|---|---|
| Consumer | Which report/page/persona uses this object? |
| Grain | What does one row mean? |
| Keys | Which columns support relationships? |
| Measures | Which calculations are approved upstream? |
| Filters | Which statuses, dates and exclusions are already applied? |
| Freshness | When is the data expected to be current? |
| Mode fit | Is the object better for Import, DirectQuery or both? |
| Owner | Who approves the definition and investigates issues? |

Day 2 will show the Power BI-side connection and refresh decisions. This
notebook builds the Databricks-side contract those decisions depend on.

## 3. Confirm the fact grain

Before defining KPI, prove what one row represents. Many BI bugs start when
someone counts lines but calls them orders.

In [ ]:
spark.sql(f"""
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(DISTINCT order_id) AS distinct_orders,
  ROUND(COUNT(*) / COUNT(DISTINCT order_id), 2) AS avg_lines_per_order
FROM {GOLD}.fact_sales_dashboard
""").show()

Expected conclusion:

- `fact_sales_dashboard` is order-line grain.
- `COUNT(*)` is not order count.
- order-level KPI must use `COUNT(DISTINCT order_id)`.

## 4. Shared KPI definitions

![KPI definition flow](../../assets/images/kpi_definition_flow.png)

Use the template in `docs/templates/kpi-dictionary.md` as the formal handoff
artifact. For Power BI developers, the key decision is not "can I write this
measure in DAX?" but "should this logic be shared upstream?"

| Logic | Keep in Databricks Gold | Keep in Power BI |
|---|---|---|
| approved revenue definition | yes, if reused across reports | only presentation-specific variants |
| row-level cleanup / status exclusion | yes | no |
| gross margin based on product cost | yes, if cost is governed upstream | no, unless cost is report-local |
| visual-only percent of selected total | no | yes |
| dynamic title / selected slicer label | no | yes |
| one-off what-if parameter | usually no | yes |

| KPI | Business definition | SQL definition | Grain caveat | Owner |
|---|---|---|---|---|
| Revenue | completed sales amount | sum completed line revenue | line revenue, completed orders only | Sales Ops |
| Gross Margin | revenue minus product cost | sum completed line margin | unit cost must be present | Finance |
| Orders | completed orders | count distinct completed order_id | not count lines | Sales Ops |
| Return Rate | returned orders / completed+returned orders | distinct returned order_id ratio | denominator agreed with business | Sales Ops |
| Data Quality Score | share of records passing checks | weighted issue score | training-specific score | Data team |

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.kpi_daily
COMMENT 'Daily KPI table for RetailHub dashboard. One row per order_date.'
AS
SELECT
  order_date,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
  ROUND(SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END), 2) AS gross_margin,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders,
  COUNT(DISTINCT CASE WHEN status = 'Returned' THEN order_id END) AS returned_orders,
  ROUND(
    100.0 * COUNT(DISTINCT CASE WHEN status = 'Returned' THEN order_id END)
    / NULLIF(COUNT(DISTINCT CASE WHEN status IN ('Completed','Returned') THEN order_id END), 0),
    2
  ) AS return_rate_pct
FROM {GOLD}.fact_sales_dashboard
GROUP BY order_date
""")

spark.sql(f"SELECT * FROM {GOLD}.kpi_daily ORDER BY order_date LIMIT 10").show()

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.kpi_monthly
COMMENT 'Monthly KPI table for Power BI executive summary. One row per year_month.'
AS
SELECT
  year_month,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
  ROUND(SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END), 2) AS gross_margin,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders,
  COUNT(DISTINCT CASE WHEN status = 'Returned' THEN order_id END) AS returned_orders,
  ROUND(
    100.0 * SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END)
    / NULLIF(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 0),
    2
  ) AS margin_rate_pct
FROM {GOLD}.fact_sales_dashboard
GROUP BY year_month
""")

spark.sql(f"SELECT * FROM {GOLD}.kpi_monthly ORDER BY year_month LIMIT 10").show()

## 5. Build BI aggregate for summary pages

![Kimball Gold model](../../assets/images/kimball_gold_model.png)

This aggregate powers Power BI summary visuals. It is deliberately not
line-grain. The point is to avoid making every report page rescan and
re-aggregate the same detail table.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_sales_dashboard_monthly
COMMENT 'BI aggregate: one row per month x region x category x channel.'
AS
SELECT
  year_month,
  customer_region,
  category,
  channel,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
  ROUND(SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END), 2) AS gross_margin,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders,
  COUNT(*) AS line_rows
FROM {GOLD}.fact_sales_dashboard
GROUP BY year_month, customer_region, category, channel
""")

spark.sql(f"SELECT * FROM {GOLD}.fact_sales_dashboard_monthly ORDER BY year_month LIMIT 10").show()

## 6. Reconciliation

Compare the daily KPI table, monthly KPI table and BI aggregate. The point is
not only to get zero diff; it is to prove that every dashboard number traces
back to the same logic.

In [ ]:
spark.sql(f"""
WITH daily_rollup AS (
  SELECT date_format(order_date, 'yyyy-MM') AS year_month, ROUND(SUM(revenue), 2) AS revenue
  FROM {GOLD}.kpi_daily
  GROUP BY date_format(order_date, 'yyyy-MM')
),
monthly AS (
  SELECT year_month, ROUND(revenue, 2) AS revenue
  FROM {GOLD}.kpi_monthly
),
aggregate_rollup AS (
  SELECT year_month, ROUND(SUM(revenue), 2) AS revenue
  FROM {GOLD}.fact_sales_dashboard_monthly
  GROUP BY year_month
)
SELECT
  m.year_month,
  d.revenue AS daily_rollup_revenue,
  m.revenue AS monthly_kpi_revenue,
  a.revenue AS aggregate_rollup_revenue,
  ROUND(d.revenue - m.revenue, 2) AS diff_daily_vs_monthly,
  ROUND(a.revenue - m.revenue, 2) AS diff_aggregate_vs_monthly
FROM monthly m
LEFT JOIN daily_rollup d ON m.year_month = d.year_month
LEFT JOIN aggregate_rollup a ON m.year_month = a.year_month
ORDER BY ABS(d.revenue - m.revenue) DESC
""").show()

## 7. Gold quality gate

Do not hide bad data in Power BI. Name it, count it and decide where it should
be fixed.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW dq_issues AS
SELECT 'missing_price' AS issue_type, COUNT(*) AS issue_count
FROM {GOLD}.fact_sales_dashboard
WHERE unit_price IS NULL
UNION ALL
SELECT 'invalid_status', COUNT(*)
FROM {GOLD}.fact_sales_dashboard
WHERE status NOT IN ('Completed', 'Cancelled', 'Returned')
UNION ALL
SELECT 'future_order_date', COUNT(*)
FROM {GOLD}.fact_sales_dashboard
WHERE order_date > current_date()
UNION ALL
SELECT 'missing_customer_dimension', COUNT(*)
FROM {GOLD}.fact_sales_dashboard
WHERE customer_region IS NULL
UNION ALL
SELECT 'missing_product_dimension', COUNT(*)
FROM {GOLD}.fact_sales_dashboard
WHERE category IS NULL
""")

spark.sql("SELECT * FROM dq_issues ORDER BY issue_count DESC").show()

In [ ]:
total_rows = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD}.fact_sales_dashboard").first()["n"]
issue_rows = spark.sql("SELECT SUM(issue_count) AS n FROM dq_issues").first()["n"] or 0
score = round(100.0 * max(total_rows - issue_rows, 0) / total_rows, 2)
print(f"Data quality score: {score}% ({issue_rows} issue flags across {total_rows} rows)")

## 8. Decision log

Use `docs/templates/decision-log.md`. A good decision log row is short but
specific.

| Date | Decision | Options considered | Chosen option | Reason | Consequence |
|---|---|---|---|---|---|
| 2026-06-23 | Power BI summary source | fact detail / monthly aggregate / Power BI measures | `gold.fact_sales_dashboard_monthly` | summary visuals do not need line grain and repeated aggregation is expensive | requires refresh job before dashboard refresh |

## 9. Import vs DirectQuery readiness

Day 2 will compare Import and DirectQuery in detail. Here we only mark whether
the Gold objects are shaped well enough for each mode.

| Gold object | Best mode | Why |
|---|---|---|
| `gold.fact_sales_dashboard_monthly` | Import baseline | small aggregate for summary visuals |
| `gold.kpi_monthly` | Import baseline | KPI cards and trends refresh predictably |
| `gold.fact_sales_dashboard` | Import or controlled drill-through | line grain, use only when detail is needed |
| `gold.fact_sales_dashboard_channel_daily` | Import or limited DirectQuery | narrow daily aggregate, useful for operational page |

DirectQuery is not bad by itself. It becomes expensive when Power BI sends many
interactive visual queries to a wide or poorly filtered object.

## Demo extension: channel daily aggregate

For a longer delivery, create `gold.fact_sales_dashboard_channel_daily` and
reconcile it against the monthly aggregate. The Day 1 workshop asks
participants to build this pattern themselves.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_sales_dashboard_channel_daily
COMMENT 'Bonus workshop aggregate: one row per order_date x channel.'
AS
SELECT
  order_date,
  channel,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
  ROUND(SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END), 2) AS gross_margin
FROM {GOLD}.fact_sales_dashboard
GROUP BY order_date, channel
""")

spark.sql(f"SELECT * FROM {GOLD}.fact_sales_dashboard_channel_daily ORDER BY order_date LIMIT 10").show()

## Self-check

In [ ]:
for table_name in ["kpi_daily", "kpi_monthly", "fact_sales_dashboard_monthly"]:
    assert spark.catalog.tableExists(f"{GOLD}.{table_name}"), f"Missing {GOLD}.{table_name}"

grain = spark.sql(f"""
SELECT COUNT(*) AS rows, COUNT(DISTINCT (year_month, customer_region, category, channel)) AS distinct_keys
FROM {GOLD}.fact_sales_dashboard_monthly
""").first()
assert grain["rows"] == grain["distinct_keys"], "Monthly BI aggregate grain is not unique"

max_diff = spark.sql(f"""
WITH daily_rollup AS (
  SELECT date_format(order_date, 'yyyy-MM') AS year_month, ROUND(SUM(revenue), 2) AS revenue
  FROM {GOLD}.kpi_daily
  GROUP BY date_format(order_date, 'yyyy-MM')
),
monthly AS (
  SELECT year_month, ROUND(revenue, 2) AS revenue
  FROM {GOLD}.kpi_monthly
)
SELECT MAX(ABS(d.revenue - m.revenue)) AS max_diff
FROM monthly m JOIN daily_rollup d ON m.year_month = d.year_month
""").first()["max_diff"]
assert max_diff < 0.01, f"Revenue reconciliation failed: {max_diff}"

print("[OK] Day 1 Gold semantic layer self-check passed")

## Summary and handoff to Day 2

Created:

- `gold.kpi_daily`
- `gold.kpi_monthly`
- `gold.fact_sales_dashboard_monthly`
- optional `gold.fact_sales_dashboard_channel_daily`

What this notebook solved:

- Databricks owns shared grain, relationships, filters and KPI definitions.
- Power BI receives curated Gold objects instead of raw transformation work.
- Aggregates exist for summary visuals.
- Detail remains available for drill-through.

Day 2 uses these objects to build the Power BI-side semantic model, connection
mode and refresh strategy.